In [1]:
import optuna
from optuna.storages.journal import JournalStorage, JournalFileBackend
import pandas as pd
import os

# === CONFIGURATION ===
MODEL_TYPE = "DeepCNNLSTM"  

DB_DIR = "C:\\Users\\kdmen\\Repos\\pers-gest-cls\\dataset\\optuna_dbs"
FILE_NAME = f"1s3w_pretrain_MOE_NumExpSweep_HPO_DeepCNNLSTM"
JOURNAL_PATH = os.path.join(DB_DIR, f"{FILE_NAME}.log")

print(f"Loading study: {FILE_NAME}")
print(f"From path: {JOURNAL_PATH}")

Loading study: 1s3w_pretrain_MOE_NumExpSweep_HPO_DeepCNNLSTM
From path: C:\Users\kdmen\Repos\pers-gest-cls\dataset\optuna_dbs\1s3w_pretrain_MOE_NumExpSweep_HPO_DeepCNNLSTM.log


In [2]:
# 2. Initialize storage
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# 3. Use the TOP-LEVEL optuna function to get summaries
summaries = optuna.get_all_study_summaries(storage)

if not summaries:
    print("No studies found in this file.")
else:
    for s in summaries:
        print(f"Internal Study Name: '{s.study_name}'")

    STUDY_NAME = s.study_name

Internal Study Name: 'pretr_hpo_NumExpSweep_DeepCNNLSTM_moe'


In [3]:
# Initialize the storage backend
storage = JournalStorage(JournalFileBackend(JOURNAL_PATH))

# Load the study
study = optuna.load_study(
    study_name=STUDY_NAME,
    storage=storage
)

print(f"Study loaded with {len(study.trials)} trials.")
print(f"Best value (Accuracy): {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Study loaded with 600 trials.
Best value (Accuracy): 0.4250
Best Hyperparameters:
  learning_rate: 0.00013500972818967897
  weight_decay: 0.001372130710017325
  dropout: 0.15000000000000002
  label_smooth: 0.1283417673027156
  batch_size: 128
  optimizer: adam
  cnn_base_filters: 128
  lstm_hidden: 256
  head_type: mlp
  MOE_placement: encoder
  MOE_gate_temperature: 3.684667833037058
  MOE_aux_coeff: 0.9093488959625315
  MOE_ctx_out_dim: 64
  MOE_ctx_hidden_dim: 128
  MOE_dropout: 0.1457069744822459
  MOE_expert_expand: 1.0465747839833475
  MOE_mlp_hidden_mult: 1.9908682100584076
  MOE_top_k: 10
  num_experts: 24


In [4]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate


In [5]:
# 1. Plot optimization history
fig_hist = plot_optimization_history(study)
fig_hist.show()


In [6]:
# 2. Plot Hyperparameter Importance
try:
    fig_imp = plot_param_importances(study)
    fig_imp.show()
except Exception as e:
    print(f"Could not plot importance (might need more trials): {e}")

In [7]:
# 3. Parallel Coordinate Plot (Visualizes high-dimensional relationships)
fig_parallel = plot_parallel_coordinate(study)
fig_parallel.show()

In [8]:
# FULL
params_to_plot = ["learning_rate", "weight_decay", "dropout", "label_smooth", "batch_size", "optimizer", "cnn_base_filters", "lstm_hidden", "head_type", 
                  "num_experts", "MOE_top_k",
                  "MOE_gate_temperature", "MOE_aux_coeff", "MOE_ctx_out_dim", "MOE_ctx_hidden_dim", "MOE_dropout", "MOE_expert_expand", "MOE_mlp_hidden_mult"]

# OPTIMIZATION
params_to_plot_OPT = ["learning_rate", "weight_decay", "dropout", "label_smooth", "batch_size", "optimizer"]

# ARCHITECTURE
params_to_plot_ARCH = ["cnn_base_filters", "lstm_hidden", "head_type"]

# MOE AUX LOSS
params_to_plot_MOE_AUX = ["MOE_gate_temperature", "MOE_aux_coeff", "MOE_dropout"]

# MOE MISC
params_to_plot_MOE_MISC = ["num_experts", "MOE_top_k", "MOE_ctx_out_dim", "MOE_ctx_hidden_dim", "MOE_expert_expand", "MOE_mlp_hidden_mult"]


In [9]:
from optuna.visualization import plot_slice


In [10]:
fig_slice = plot_slice(study, params=params_to_plot_OPT)
fig_slice.show()

In [11]:
fig_slice = plot_slice(study, params=params_to_plot_ARCH)
fig_slice.show()

In [12]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_AUX)
fig_slice.show()

In [13]:
fig_slice = plot_slice(study, params=params_to_plot_MOE_MISC)
fig_slice.show()

## Best Hyperparameter Ranges

- learning_rate: lower better. 0.0001-0.0002. Could maybe go lower (0.0001 was the lowest I tested...)
- weight_decay: no real trend... best trial is 0.001 but most trials are <0.0001
- dropout: no real trend... anything between 0.05-0.25...
- label_smooth: no real trend... best 4 trials were all 0.09-0.15
- batch_size: it chose 128 which does have the best trials ig
- optimizer: kind of a wash. Adam is a little better
- cnn_base_filters: 128 best, maybe could go higher
- lstm_hidden: 128 is probably best? 256 is okay. 64 and 512 are bad
- head_type: no real trend... MLP maybe a little better?
- MOE_placement: encoder
- MOE_gate_temperature: 2-6 
- MOE_aux_coeff: 0.2-1 (1 was the highest tested)
- MOE_ctx_out_dim: 128
- MOE_ctx_hidden_dim: 64? 32 okay. 128 bad.
- MOE_dropout: 0.12-0.17
- MOE_expert_expand: 1
- MOE_mlp_hidden_mult: 1
- MOE_top_k: 1 and 2 are bad. 8 and 10 (and 3...) do best
- num_experts: 10 is bad, goes up until peaking at 32, then does down until 60, then start to go back up until 120 (the highest tested)

In [14]:
trials_df = study.trials_dataframe()

# Extract the custom user attributes into the dataframe
trials_df['mean_pretrain_val_acc'] = [t.user_attrs.get('mean_pretrain_val_acc') for t in study.trials]
trials_df['fold_mean_accs'] = [t.user_attrs.get('fold_mean_accs') for t in study.trials]

# Filter for relevant columns and sort by best performance
results_summary = trials_df[['number', 'value', 'mean_pretrain_val_acc', 'datetime_start', 'duration']].sort_values(by='value', ascending=False)
results_summary.head(10)

,number,value,mean_pretrain_val_acc,datetime_start,duration
206,206,0.4250,None,2026-04-02 19:48:43.517585,0 days 00:00:16.122839
87,87,0.3875,None,2026-04-02 19:35:37.686651,0 days 00:00:11.052138
236,236,0.3875,None,2026-04-02 19:50:56.238423,0 days 00:04:25.746795
221,221,0.3750,None,2026-04-02 19:49:59.499251,0 days 00:00:52.007465
247,247,0.3625,None,2026-04-02 19:51:38.811590,0 days 00:01:08.019984
148,148,0.3625,None,2026-04-02 19:42:50.604117,0 days 00:00:56.858292
543,543,0.3625,None,2026-04-02 20:41:41.476200,0 days 00:01:31.338570
279,279,0.3500,None,2026-04-02 19:57:09.552438,0 days 00:01:00.906616
244,244,0.3500,None,2026-04-02 19:51:18.357072,0 days 00:05:14.223856
270,270,0.3500,None,2026-04-02 19:56:13.583660,0 days 00:01:07.363450
